In [ ]:
import os
import sys
from hydra import initialize, compose
from omegaconf import OmegaConf
from pathlib import Path
from loguru import logger
import pandas as pd

sys.path.insert(1, str(Path.cwd().parent.resolve()))

from utils import data_handling

In [ ]:
hydra_config_path: Path = Path("workspace_dani/conf")
hydra_config_path_relative = hydra_config_path.relative_to(".")

with initialize(version_base=None, config_path=str("../../conf"), job_name="config_data"):
    # cfg = compose(overrides=["+table=apache_patient_results"])
    cfg = compose(config_name="config_data.yaml",
                  overrides=["data=drop_columns", "table@data.table=apache_aps_var"],
                  return_hydra_config=True)

    print(OmegaConf.to_yaml(cfg, resolve=False))


In [ ]:
# Todo: .yaml files mit drop columns anpassen

In [ ]:
dropping_columns: bool = cfg.data.table.dropping_columns

In [ ]:
column_to_drop: list = cfg.data.table.columns_to_drop

In [ ]:
column_to_keep: list = cfg.data.table.columns_to_keep

In [ ]:
cwd_path: Path = Path.cwd()
project_root_path: Path = cwd_path.parents[2]
parquet_data_path: Path = project_root_path / Path(cfg.data.table.paths.table_location)


In [ ]:
df_data = data_handling.load_parquet_table(parquet_path=parquet_data_path, verbose=False)


In [ ]:
df_data.head()


In [ ]:
len(df_data.columns)

In [ ]:
len(column_to_keep)

In [ ]:
if dropping_columns:
    logger.info("Dropping columns!")
    if len(column_to_drop) != 0:
        df_out = df_data.drop(columns=column_to_drop)
    elif len(column_to_keep) != 0:
        df_out = df_data[column_to_keep]
    else:
        df_out = None
        logger.error(f"There's something wrong {dropping_columns = } but no columns where specified!")
    
else:
    df_out = df_data
    logger.info(f"Won't drop any columns from table! {dropping_columns = }")

In [ ]:
df_out